In [1]:
import os
os.chdir('../../../../..')

In [2]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN, KMeans
from kmedoids import KMedoids

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [3]:
qm9 = QM9Dataset(limit=5000, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"], add_morgan_fingerprint=True)
molecules = qm9.get_molecules()
df = qm9.load()

2026-04-21 15:21:35.163 | INFO     | src.datasets:load:500 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-21 15:21:35.405 | INFO     | src.datasets:_sample_qm9_df:692 - QM9 sampling complete: strategy=stratified, requested_limit=5000, returned_rows=5000.
2026-04-21 15:21:35.405 | INFO     | src.datasets:_add_requested_descriptors:129 - Applying requested QM9 descriptors to sampled dataframe (rows=5000).
2026-04-21 15:21:35.405 | INFO     | src.features:compute_morgan_fingerprints:76 - Computing Morgan Fingerprints (Radius=3, Size=2048)...
2026-04-21 15:21:44.628 | INFO     | src.datasets:_add_requested_descriptors:152 - Added descriptor column(s): ['morgan_fingerprint']
2026-04-21 15:22:08.471 | SUCCESS  | src.datasets:get_molecules:1185 - Saved 5000 molecules to data/QM9/qm9_subset.xyz (failed: 0, requested: 5000).
2026-04-21 15:22:08.473 | INFO     | src.datasets:load:500 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-0

In [4]:
plot_molecules_with_py3dmol(molecules[0:3])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
X = np.array(df['morgan_fingerprint'].to_list())

In [6]:
dist_matrix = qm9.get_distance_matrix(descriptor="morgan", dist_type="jaccard", force_calculate=True)

2026-04-21 15:22:59.331 | INFO     | src.datasets:get_distance_matrix:1016 - Calculating distance matrix for morgan using jaccard distance.
2026-04-21 15:23:17.627 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_morgan_jaccard.npy


In [7]:
n = 10
triu_indices = np.triu_indices_from(dist_matrix, k=1)
distances = dist_matrix[triu_indices]
molecule_pairs = list(zip(triu_indices[0], triu_indices[1]))
smallest_indices = np.argsort(distances)[:n]
closest_pairs = [molecule_pairs[i] for i in smallest_indices]
print("Closest molecule pairs (indices):", closest_pairs)
mols = [(molecules[idx1], molecules[idx2]) for idx1, idx2 in closest_pairs]

Closest molecule pairs (indices): [(np.int64(4966), np.int64(4967)), (np.int64(593), np.int64(4376)), (np.int64(413), np.int64(4473)), (np.int64(143), np.int64(3070)), (np.int64(236), np.int64(960)), (np.int64(128), np.int64(593)), (np.int64(448), np.int64(2336)), (np.int64(617), np.int64(1931)), (np.int64(23), np.int64(146)), (np.int64(128), np.int64(4376))]


In [8]:
plot_molecules_with_py3dmol(mols[2])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Determining the best number of clusters for each clustering method

In [5]:
out = evaluate_distance_matrix_clustering_sweep(
    dist_matrix=dist_matrix,
    fingerprint="morgan",
    distance_metric="tanimoto",
    dataset_name="qm9",
)

NameError: name 'dist_matrix' is not defined

# Hiercical Clustering on Distance Matrix

In [30]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=3, linkage='average')
labels_hier = model_hier.fit_predict(dist_matrix)
df = df.with_columns(labels_hier=labels_hier)
print(np.unique(labels_hier, return_counts=True))

(array([0, 1, 2]), array([ 938,    5, 4057]))


In [29]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 't-SNE')

Running t-SNE dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_t-SNE_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_t-SNE_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - t-SNE Clustering'}, settings={'map': {'x': {'property': 't-SNE_1'}, 'y'…

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="morgan",
    distance_metric="tanimoto",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_hier,
    clustering_method="hierarchical"
)

2026-04-21 13:37:17.864 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/tanimoto/morgan/pca_hierarchical_projection.png


{'coords': array([[ 1.21527925,  1.71274172],
        [ 1.98816896, -1.69302976],
        [-0.2607709 , -0.32863902],
        ...,
        [ 2.57937249, -1.84158736],
        [-0.4344677 , -0.38770645],
        [ 2.69716913, -1.57512399]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/tanimoto/morgan/pca_hierarchical_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/tanimoto/morgan'),
 'clustering_method': 'hierarchical'}

In [31]:
average_numeric_by_cluster(df, "labels_hier")

shape: (3, 62)
┌─────────────┬───────┬─────────────────────┬───────────┬────────────┬────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬────────────────────┬──────────────┬─────────────┬──────────────────┬──────────────┬──────────────────┐
│ labels_hier ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp   ┆ tpsa    ┆ election_affinity

labels_hier,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,pct_aliphatic_ring,pct_aromatic,pct_acyclic,unique_scaffolds,top_scaffold,top_scaffold_pct
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,f64
0,938,1.862363,15.585288,121.782516,0.229211,49.892324,0.671086,12.964864,8.778252,1.260128,0.929638,2.033214,2.025586,0.035884,0.658612,0.305504,1.030917,2.872068,6.098081,0.212154,3.39339,1.707889,6.34435,29.316631,1.259139,0.015991,0.113006,0.175906,0.242004,0.037313,0.0,0.02452,0.022388,0.234542,0.015991,3.464819,3.104825,72.444392,-6.14693,-0.349829,5.797133,1143.219847,3.316243,-11505.440162,-11505.224837,-11505.199167,-11506.33467,29.06617,-68.072215,-68.457573,-68.832413,-63.475686,3.903549,1.444652,1.047985,11.727079,88.272921,0.0,344,"""c1cc[nH]c1""",8.102345
1,5,2.1717,18.6,104.8,1.0,10.8,0.967794,12.821829,7.4,1.4,0.0,2.0401,0.0,0.0,0.057143,0.942857,0.0,1.2,6.2,0.0,0.4,5.8,5.4,40.4,1.253312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.2,0.0,1.2,0.82556,68.16,-6.837133,1.986975,8.824653,819.3302,4.540443,-9064.377832,-9064.192773,-9064.16709,-9065.214746,26.5914,-75.626142,-76.15788,-76.610278,-70.273891,3.732526,2.92259,1.857912,100.0,0.0,0.0,5,"""C1CCCOCC1""",20.0
2,4057,2.13382,18.593049,123.048558,0.065319,33.767069,0.897557,12.829491,8.789007,1.728617,0.008381,2.066645,2.373675,0.071287,0.138372,0.790342,0.893024,1.829184,6.604387,0.473996,0.845945,5.235642,6.356175,39.841509,1.26328,0.0,0.39019,0.000739,0.104757,0.147153,0.002218,0.042889,0.146167,0.585408,0.0,2.233424,2.600288,75.80649,-6.62572,0.449962,7.075683,1203.457074,4.228471,-11107.42893,-11107.19377,-11107.168095,-11108.34287,32.2728,-77.980447,-78.461892,-78.914075,-72.547039,3.290189,1.389039,1.1323,86.295292,0.838058,12.86665,1226,"""Acyclic""",12.86665


# KMedoids

In [35]:
model_km = KMedoids(n_clusters=20, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
df = df.with_columns(labels_km=labels_km)

In [37]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [16]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="morgan",
    distance_metric="tanimoto",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_km,
    clustering_method="kmedoids"
)

2026-04-21 13:37:43.075 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/tanimoto/morgan/pca_kmedoids_projection.png


{'coords': array([[ 1.21527925,  1.71274172],
        [ 1.98816896, -1.69302976],
        [-0.2607709 , -0.32863902],
        ...,
        [ 2.57937249, -1.84158736],
        [-0.4344677 , -0.38770645],
        [ 2.69716913, -1.57512399]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/tanimoto/morgan/pca_kmedoids_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/tanimoto/morgan'),
 'clustering_method': 'kmedoids'}

In [36]:
average_numeric_by_cluster(df, "labels_km")

shape: (20, 64)
┌───────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────────┬────────────────────┬──────────────┬─────────────┬──────────────────┬──────────────┬──────────────────┐
│ labels_km ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    

labels_km,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,kmeans_labels,pct_aliphatic_ring,pct_aromatic,pct_acyclic,unique_scaffolds,top_scaffold,top_scaffold_pct
u64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,f64
0,207,1.865508,15.111111,121.15942,0.193237,48.541063,0.68298,12.952534,8.768116,1.169082,0.942029,2.021992,1.541063,0.028502,0.75245,0.219048,0.985507,2.425121,6.362319,0.173913,3.922705,1.256039,6.115942,28.454106,1.258719,0.009662,0.038647,0.057971,0.140097,0.111111,0.0,0.009662,0.057971,0.057971,0.019324,3.415459,3.867334,72.978309,-6.18737,-0.743436,5.44392,1118.241479,3.171583,-11479.282988,-11479.071961,-11479.046285,-11480.172323,28.501821,-67.349993,-67.721357,-68.084003,-62.887604,3.713469,1.501062,1.048065,0.280193,0.937198,10.628019,89.371981,0.0,110,"""c1cc[nH]c1""",12.077295
1,328,2.204771,19.170732,123.414634,0.143293,26.0,0.927649,12.810231,8.807927,2.564024,0.018293,2.141103,1.948171,0.011106,0.075783,0.913111,0.780488,1.643293,7.115854,0.079268,0.539634,6.207317,6.341463,42.292683,1.272126,0.0,0.405488,0.0,0.07622,0.021341,0.0,0.0,0.02439,0.960366,0.0,1.981707,1.908663,75.749939,-6.486191,1.231124,7.717423,1092.384697,4.447667,-11012.765504,-11012.551144,-11012.525513,-11013.648274,30.760351,-79.877365,-80.4019,-80.868937,-74.246604,3.172207,1.596721,1.328423,1.926829,1.905488,98.170732,1.829268,0.0,230,"""C1CO1""",2.743902
2,235,1.99417,18.668085,122.923404,0.106383,48.451064,0.832264,12.924817,8.685106,0.268085,0.059574,1.921035,4.502128,0.1978,0.092796,0.709404,1.238298,2.4,5.191489,1.182979,0.497872,4.306383,6.948936,37.604255,1.238438,0.0,0.855319,0.004255,0.119149,0.085106,0.0,0.021277,0.07234,0.306383,0.0,2.697872,3.266994,75.359149,-7.135022,0.307639,7.442661,1485.839164,4.22598,-11264.661102,-11264.389545,-11264.363898,-11265.633789,35.113196,-77.052778,-77.500706,-77.954824,-71.66141,3.654449,1.036803,0.845903,1.880851,1.93617,20.425532,5.957447,73.617021,24,"""Acyclic""",73.617021
3,138,2.171075,18.528986,122.565217,0.231884,25.014493,0.930173,12.811838,8.753623,2.355072,0.043478,2.130055,0.913043,0.030418,0.082382,0.887201,0.384058,1.76087,6.731884,0.210145,0.543478,5.905797,6.144928,40.391304,1.275055,0.0,0.094203,0.0,0.065217,0.028986,0.0,0.014493,0.028986,1.297101,0.0,2.094203,2.085222,74.267899,-6.58202,1.120518,7.702715,1079.453686,4.276063,-11054.514946,-11054.310925,-11054.28522,-11055.391357,29.13363,-77.847612,-78.357681,-78.808219,-72.401012,3.463507,1.615016,1.312602,1.869565,1.746377,95.652174,4.347826,0.0,123,"""C1OC2CC1C2""",2.173913
4,248,2.167721,20.197581,125.967742,0.133065,30.939516,0.933857,12.846795,8.899194,2.016129,0.024194,2.08972,3.346774,0.010753,0.061345,0.927903,1.03629,1.883065,7.024194,0.072581,0.403226,6.310484,6.689516,43.866935,1.26124,0.0,0.709677,0.004032,0.048387,0.008065,0.004032,0.012097,0.024194,0.846774,0.0,2.112903,2.132008,77.934153,-6.509655,1.372529,7.882283,1260.357135,4.708927,-11294.028151,-11293.781313,-11293.755655,-11294.952015,34.104089,-82.625352,-83.156919,-83.650362,-76.727011,3.131133,1.270181,1.06243,1.91129,1.891129,97.580645,2.419355,0.0,118,"""C1COC1""",8.064516
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
15,179,1.951371,18.614525,120.340

# Spectral

In [18]:
model_spectral = SpectralClustering(
                n_clusters=3,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)
print(np.unique(labels_spectral, return_counts=True))

(array([0, 1, 2], dtype=int32), array([4998,    1,    1]))


In [19]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [20]:
labels_km

array([0, 2, 1, ..., 2, 2, 2], shape=(5000,), dtype=uint64)

In [21]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="morgan",
    distance_metric="tanimoto",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_spectral,
    clustering_method="spectral"
)

2026-04-21 13:38:32.951 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/tanimoto/morgan/pca_spectral_projection.png


{'coords': array([[ 1.21527925,  1.71274172],
        [ 1.98816896, -1.69302976],
        [-0.2607709 , -0.32863902],
        ...,
        [ 2.57937249, -1.84158736],
        [-0.4344677 , -0.38770645],
        [ 2.69716913, -1.57512399]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/tanimoto/morgan/pca_spectral_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/tanimoto/morgan'),
 'clustering_method': 'spectral'}

In [22]:
average_numeric_by_cluster(df, "labels_spectral")

shape: (3, 61)
┌─────────────────┬───────┬─────────────────────┬───────────┬────────────┬────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬────────────────────┬──────────────┬─────────────┐
│ labels_spectral ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp   ┆ tpsa    ┆ election_affinity ┆ ionization_energ

labels_spectral,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,labels_km,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,4998,2.08302,18.029412,122.794518,0.096839,36.770108,0.855166,12.854822,8.785714,1.640656,0.181072,2.060371,2.306523,0.06456,0.235822,0.699618,0.918167,2.02421,6.509404,0.42437,1.323129,4.57483,6.352941,37.869748,1.262488,0.003001,0.337735,0.033613,0.130452,0.126451,0.001801,0.039416,0.122849,0.520408,0.003001,2.463385,2.693401,75.169104,-6.53571,0.301639,6.837356,1191.783601,4.057804,-11180.138482,-11179.907082,-11179.881407,-11181.048715,31.666954,-76.121031,-76.584512,-77.022203,-70.844529,3.405078,1.4011,1.11727,1.62405,1.020008,72.328932,17.226891,10.444178
1,1,1.545455,11.0,123.0,0.0,49.0,0.638198,13.138502,9.0,1.0,1.0,2.0,0.0,0.2,0.8,0.0,0.0,3.0,4.0,1.0,4.0,0.0,6.0,17.0,1.304038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.9366,67.389999,-7.964772,-2.468073,5.4967,1155.412476,1.827653,-12401.580078,-12401.399414,-12401.374023,-12402.435547,24.056,-55.645657,-55.889771,-56.146702,-52.312183,6.03786,1.02362,0.87524,0.0,0.0,0.0,100.0,0.0
2,1,2.181818,22.0,114.0,1.0,20.0,0.94794,12.855371,8.0,1.0,0.0,2.0,2.0,0.0,0.0,1.0,1.0,1.0,7.0,0.0,0.0,7.0,7.0,48.0,1.246098,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.2696,78.0,-7.110335,2.166026,9.276361,1172.364502,5.510877,-9530.21875,-9529.992188,-9529.966797,-9531.104492,32.279999,-87.982719,-88.603767,-89.143585,-81.631775,3.87012,1.30822,1.05298,2.0,2.0,100.0,0.0,0.0


# DBSCAN 

In [33]:
model_db = DBSCAN(
    eps=0.8,
    min_samples=2,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db, return_counts=True))

(array([-1,  0,  1,  2,  3,  4,  5,  6,  7]), array([  20, 4961,    2,    2,    4,    4,    3,    2,    2]))


In [36]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [37]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="morgan",
    distance_metric="tanimoto",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_db,
    clustering_method="dbscan"
)

2026-04-21 13:44:54.711 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/tanimoto/morgan/pca_dbscan_projection.png


{'coords': array([[ 1.21527925,  1.71274172],
        [ 1.98816896, -1.69302976],
        [-0.2607709 , -0.32863902],
        ...,
        [ 2.57937249, -1.84158736],
        [-0.4344677 , -0.38770645],
        [ 2.69716913, -1.57512399]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/tanimoto/morgan/pca_dbscan_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/tanimoto/morgan'),
 'clustering_method': 'dbscan'}

In [38]:
clusters = average_numeric_by_cluster(df, "labels_db")
clusters.show(limit=25)

shape: (9, 63)
┌───────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬─────────────────┬───────────────┬────────────────────┬──────────────┬─────────────┐
│ labels_db ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_aff

labels_db,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,labels_km,labels_spectral,kmeans_labels,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-1,20,1.982317,14.0,121.35,-0.35,54.1,0.434298,13.106688,8.8,1.95,0.9,2.124564,0.7,0.025,0.706131,0.268869,1.1,2.95,6.35,0.1,3.1,1.55,5.9,28.2,1.271156,0.0,0.0,0.05,0.55,0.0,0.05,0.0,0.0,0.05,0.0,4.05,4.796605,70.149,-6.192223,-0.816478,5.375881,957.25625,2.857846,-11463.577148,-11463.397021,-11463.371191,-11464.423584,25.4675,-61.804194,-62.163584,-62.497659,-57.60794,4.110221,1.714434,1.244005,0.7,0.1,0.0,1.05,40.0,60.0,0.0
0,4961,2.084161,18.061681,122.792985,0.097359,36.629309,0.857806,12.852904,8.785124,1.640194,0.176577,2.060108,2.316872,0.064981,0.231657,0.703363,0.916348,2.017134,6.512598,0.427333,1.305987,4.60129,6.354969,37.953034,1.262427,0.003024,0.339649,0.032856,0.128603,0.127192,0.001613,0.03971,0.123765,0.523282,0.003024,2.450514,2.684956,75.204896,-6.538606,0.31217,6.850783,1193.427261,4.067718,-11175.201364,-11174.969623,-11174.943949,-11176.112005,31.709364,-76.236466,-76.700851,-77.139371,-70.951255,3.402927,1.398659,1.116594,1.631324,1.026204,0.000605,1.483773,72.5257,16.952227,10.522072
1,2,1.807692,13.0,116.0,0.5,62.5,0.393377,13.057106,8.5,1.0,0.0,2.0,1.0,0.0,1.0,0.0,1.0,2.5,5.5,0.0,4.5,0.0,6.0,23.5,1.2731,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,4.0,2.71605,71.445,-6.842303,-2.404126,4.438177,873.491882,2.572809,-10973.925781,-10973.743652,-10973.718262,-10974.776855,24.8405,-58.418196,-58.737267,-59.045641,-54.548006,3.088295,2.098375,1.37873,0.0,0.0,0.0,1.0,100.0,0.0,0.0
2,2,2.17381,14.5,115.0,0.5,30.0,0.694857,12.813877,8.5,3.0,1.0,2.258333,0.5,0.0,0.5,0.5,1.5,1.5,8.0,0.0,3.0,3.0,6.0,31.5,1.27876,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.5,3.40095,69.690002,-5.608267,0.355109,5.962015,858.515137,3.067961,-10317.470703,-10317.305176,-10317.279297,-10318.294922,24.293,-64.702887,-65.096088,-65.443037,-60.390657,4.60368,1.82866,1.477235,2.0,0.0,0.0,0.5,0.0,100.0,0.0
3,4,1.821487,17.0,120.0,1.0,28.0,0.705207,12.749954,9.0,1.0,1.0,2.0,0.0,0.0,1.0,0.0,1.0,1.0,8.0,0.0,7.0,0.0,6.0,31.0,1.239993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.1726,88.3675,-5.4001,-0.851036,4.548383,1044.925995,3.768192,-10363.406494,-10363.201904,-10363.176514,-10364.27124,29.51025,-74.854586,-75.305737,-75.716942,-69.811499,2.412133,1.890455,1.161532,0.0,1.25,0.0,1.0,0.0,100.0,0.0
4,4,2.197161,13.5,130.75,-0.5,59.0,0.6213,13.334929,9.0,1.0,0.0,2.0,2.25,0.0,0.566667,0.433333,1.75,3.75,5.25,0.0,2.25,1.75,6.5,29.75,1.254497,0.0,0.5,0.0,0.75,0.0,0.0,0.0,0.0,0.0,0.0,5.0,2.2692,70.3325,-6.768152,-2.274872,4.49328,1114.984802,2.536353,-13114.937988,-13114.720703,-13114.694824,-13115.845703,28.5005,-57.659982,-57.962408,-58.283625,-53.640123,2.938978,1.52925,0.983613,2.0,0.0,0.0,1.0,100.0,0.0,0.0
5,3,1.530303,11.333333,125.666667,0.0,71.666667,0.326196,13.221801,9.0,2.0,2.0,2.176768,1.0,0.0,1.0,0.0,1.0,5.666667,4.333333,0.0,3.333333,0.0,6.666667,17.333333,1.296509,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.666667,2.651133,58.856668,-6.340253,-0.827226,5.51212,939.659261,1.973016,-13048.005534,-13047.839518,-13047.814128,-13048.835286,23.265999,-52.032955,-52.303891,-52.569392,-48.552929,4.985837,1.521013,1.143567,0.0,0.0,0.0,1.0,0.0,100.0,0.

# KMeans on embedding

In [27]:
df

mol_id,formula,smiles,canonical_smiles,scaffold_smiles,selfies,functional_groups,num_atoms,structure_class,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,morgan_fingerprint,labels_hier,labels_km,labels_spectral,labels_db
str,str,str,str,str,str,str,i64,str,i64,i64,i64,f64,f64,i64,i64,i64,f64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,list[i8],i64,u64,i32,i64
"""qm9_89""","""C3H5NO""","""[H]N1C([H])([H])C(=O)C1([H])[H…","""[H]N1C([H])([H])C(=O)C1([H])[H…","""O=C1CNC1""","""[H][N][C][Branch1][C][H][Branc…","""ketone""",10,"""Aliphatic Ring""",71,0,29,0.761845,12.992522,5,1,0,2.0,0,0.0,0.333333,0.666667,1,2,4,0,1,2,4,20,1.261328,0,0,0,0,0,0,0,1,0,0,2,2.5257,39.34,-6.631414,-0.702054,5.929361,359.152008,2.167387,-6726.387695,-6726.26123,-6726.235352,-6727.135742,16.083,-40.819069,-41.077904,-41.309227,-38.114258,11.54185,4.90368,3.63309,"[0, 0, … 0]",2,0,0,-1
"""qm9_132""","""C4H10O""","""[H]C([H])([H])C([H])([H])OC([H…","""[H]C([H])([H])C([H])([H])OC([H…","""Acyclic""","""[H][C][Branch1][C][H][Branch1]…","""ether""",15,"""Acyclic""",74,1,9,0.936769,12.976237,5,0,0,1.866667,4,0.0,0.0,1.0,0,1,4,0,0,4,6,31,1.209514,0,0,0,0,0,0,0,0,1,0,1,0.9301,50.619999,-6.756587,2.53338,9.289967,631.081787,3.704885,-6355.161621,-6354.975098,-6354.949219,-6355.981445,22.881001,-56.966496,-57.358181,-57.718079,-52.897472,17.9608,2.23958,2.09578,"[0, 0, … 0]",2,2,0,0
"""qm9_139""","""C3H4N2""","""[H]N1C([H])([H])[C@]1([H])C#N""","""[H]N1C([H])([H])[C@]1([H])C#N""","""C1CN1""","""[H][N][C][Branch1][C][H][Branc…","""""",9,"""Aliphatic Ring""",68,0,45,0.444793,13.026985,5,1,0,2.0,0,0.333333,0.0,0.666667,1,2,3,1,0,2,4,17,1.264596,0,0,0,0,0,0,0,0,0,0,2,2.8377,40.209999,-7.55116,0.206807,7.757967,386.555695,1.880987,-6152.69043,-6152.564453,-6152.538574,-6153.435547,15.572,-38.04356,-38.264439,-38.47002,-35.623112,17.103571,3.50921,3.35299,"[0, 0, … 0]",2,1,0,0
"""qm9_144""","""C3H5NO""","""[H]C(=O)N1C([H])([H])C1([H])[H…","""[H]C(=O)N1C([H])([H])C1([H])[H…","""C1CN1""","""[H][C][=Branch1][C][=O][N][C][…","""amide""",10,"""Aliphatic Ring""",71,0,20,0.761845,12.992522,5,1,0,2.0,0,0.0,0.333333,0.666667,0,1,4,0,1,2,4,20,1.261934,0,0,0,0,1,0,0,0,0,0,2,3.2517,40.77,-6.952509,-0.035375,6.917135,383.601288,2.16589,-6726.451172,-6726.317871,-6726.291992,-6727.207031,16.197001,-40.882473,-41.134586,-41.36591,-38.185608,14.7497,3.94953,3.42803,"[0, 0, … 1]",2,2,0,0
"""qm9_149""","""C4H8O""","""[H]OC([H])([H])C1([H])C([H])([…","""[H]OC([H])([H])C1([H])C([H])([…","""C1CC1""","""[H][O][C][Branch1][C][H][Branc…","""alcohol""",13,"""Aliphatic Ring""",72,0,20,0.964858,12.880514,5,1,0,2.0,2,0.0,0.0,1.0,1,1,4,0,0,4,5,27,1.240861,0,1,0,0,0,0,0,0,0,0,1,1.3201,46.009998,-7.077682,2.1497,9.224659,468.622894,3.110261,-6321.884277,-6321.729004,-6321.703125,-6322.664062,20.059,-50.915638,-51.26128,-51.569748,-47.386292,12.75002,3.20516,2.88061,"[0, 0, … 0]",2,2,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""qm9_130677""","""C4H5F3O2""","""[H]C(=O)O[C@]([H])(C([H])([H])…","""[H]C(=O)O[C@]([H])(C([H])([H])…","""Acyclic""","""[H][C][=Branch1][C][=O][O][C@]…","""ether,halogen""",14,"""Acyclic""",142,1,26,1.567518,13.752707,9,0,0,1.857143,2,0.0,0.25,0.75,0,2,4,0,1,3,5,28,1.279761,0,0,0,0,0,0,0,0,1,0,5,3.6403,52.540001,-7.93484,-0.185037,7.752524,1188.842651,2.583122,-16472.59375,-16472.349609,-16472.324219,-

In [32]:
embedding = np.array(df['morgan_fingerprint'].to_list())
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(embedding)
df = df.with_columns(kmeans_labels=kmeans_labels)

In [33]:
create_chemiscope_viewer(df, embedding, kmeans_labels, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [34]:
average_numeric_by_cluster(df, "kmeans_labels")

shape: (3, 63)
┌───────────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬────────────────────┬──────────────┬─────────────┬──────────────────┬──────────────┬──────────────────┐
│ kmeans_labels ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa   

kmeans_labels,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,pct_aliphatic_ring,pct_aromatic,pct_acyclic,unique_scaffolds,top_scaffold,top_scaffold_pct
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,f64
0,830,2.105239,17.074699,123.438554,-0.090361,39.06747,0.910328,12.804751,8.838554,1.853012,0.024096,2.088194,1.36506,0.027057,0.286211,0.686732,0.818072,2.166265,6.679518,0.168675,1.751807,4.287952,5.854217,36.086747,1.275144,0.0,0.251807,0.00241,0.066265,0.3,0.001205,0.16506,0.504819,0.501205,0.0,2.63012,3.275423,72.455964,-6.585005,-0.491854,6.093187,1088.079534,3.757598,-11503.780035,-11503.561495,-11503.535813,-11504.672572,30.179349,-73.905888,-74.345462,-74.758601,-68.881489,3.015427,1.548927,1.188257,1.881928,97.590361,2.409639,0.0,477,"""O=C1CCN1""",3.373494
1,940,1.865403,15.655319,121.693617,0.25,49.782979,0.666446,12.972761,8.770213,1.252128,0.939362,2.031641,2.08617,0.031578,0.671562,0.296861,1.080851,2.809574,6.182979,0.181915,3.457447,1.685106,6.354255,29.525532,1.257965,0.015957,0.115957,0.176596,0.247872,0.040426,0.0,0.006383,0.015957,0.196809,0.015957,3.445745,3.089867,72.762266,-6.110771,-0.348743,5.762037,1146.861994,3.339204,-11473.852783,-11473.636707,-11473.611037,-11474.747967,29.169919,-68.26095,-68.64824,-69.024881,-63.645832,3.914128,1.442241,1.04485,0.189362,10.744681,89.255319,0.0,330,"""c1cc[nH]c1""",8.510638
2,3230,2.140506,18.964706,122.946749,0.100619,32.391331,0.895875,12.833454,8.776471,1.698762,0.000929,2.061545,2.611765,0.083817,0.096165,0.820017,0.896285,1.759133,6.560062,0.560681,0.59226,5.488854,6.480805,40.752941,1.26056,0.0,0.424458,0.0,0.112693,0.106811,0.002477,0.016718,0.055728,0.619195,0.0,2.134675,2.428095,76.565198,-6.64733,0.694535,7.341864,1231.487903,4.343835,-11011.363742,-11011.124595,-11011.098921,-11012.282878,32.773742,-78.975034,-79.466813,-79.928555,-73.441543,3.358019,1.350994,1.120009,1.974923,83.74613,0.092879,16.160991,786,"""Acyclic""",16.160991
